# Langfuse Smoke Test — Compliance Pipeline

**Purpose:** Fire 10 synthetic alerts through the compliance pipeline and verify that:
1. The API responds correctly for each alert (status 200, risk_score, decision present).
2. Langfuse received and indexed the traces (verified via `/api/v1/metrics/summary`).

**Run once after deploy.** Requires the API to be running at `http://api:8000`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [ ]:
import json
import time
import requests
import os
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "http://api:8000"

In [ ]:
with open("langfuse_smoke_test_alerts.json") as f:
    alerts = json.load(f)

print(f"Loaded {len(alerts)} test alerts")

In [ ]:
responses = []

for alert in alerts:
    alert_id = alert["alert_id"]
    payload = {
        "customer_id": alert["customer_id"],
        "alert_type": alert["alert_type"],
        "description": alert["description"],
    }
    r = requests.post(f"{BASE_URL}/api/v1/alerts/{alert_id}/analyze", json=payload)
    body = r.json() if r.ok else {}
    print(
        f"{alert_id} | status={r.status_code} "
        f"| decision={body.get('decision')} "
        f"| risk_score={body.get('risk_score')}"
    )
    responses.append((r.status_code, body))

In [ ]:
assert all(status == 200 for status, _ in responses), "Some requests did not return 200"
assert all(body.get("risk_score") is not None for _, body in responses), "Some responses missing risk_score"
assert all(body.get("decision") is not None for _, body in responses), "Some responses missing decision"

print("\u2713 All assertions passed")

## Wait... until LangFuse ingests everything

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

from metrics_lib import get_metrics

summary = get_metrics(from_date=None, to_date=None)

In [ ]:
print(json.dumps(summary, indent=2))

In [ ]:
assert summary.get("total_traces") is not None, "No traces found in Langfuse"
print("✓ Langfuse connected")

## Verify in Langfuse Dashboard

Check the Langfuse dashboard at https://us.cloud.langfuse.com for traces named **`compliance-pipeline`**.

Each alert should appear as one trace with:
- 3 child observations: `investigador`, `risk_analyzer`, `decision_agent`
- 4 scores: `risk_score`, `escalation_decision`, `auto_dismissed`, `confidence`

Alerts that triggered auto-escalation (risk_score ≥ 9) will have only 2 child observations
(`investigador` and `risk_analyzer`) and will be missing the `decision_agent` span.